# 🔬 MediaTruth — Custom Model Training

**This notebook trains two production-grade models for the MediaTruth AI forensics platform:**

| Model | Architecture | Purpose | Dataset |
|---|---|---|---|
| Deepfake Detector | EfficientNet-B5 | Detect AI-generated face manipulation | CIFAKE + 140k Faces |
| GAN Detector | ResNet-50 (CNNDetect) | Detect GAN-synthesized images | CIFAKE + ProGAN dataset |

**Output:** Two `.pth` files ready to drop into `backend/models/weights/`

---

## ⚡ Quick Start
1. Go to **Runtime → Change runtime type → GPU (T4 or A100)**
2. Run all cells (Runtime → Run all)
3. Download the exported `.pth` files at the end
4. Place them in `MediaTruth/backend/models/weights/`

> **Estimated training time:** ~45 min (T4 GPU) | ~15 min (A100 GPU)

## 📦 Step 1: Install Dependencies

In [ ]:
# Install required packages
!pip install -q timm kaggle albumentations scikit-learn matplotlib seaborn tqdm

import torch
import sys

# Verify GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Device: {device}')
if torch.cuda.is_available():
    print(f'   GPU: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️  No GPU found! Go to Runtime → Change runtime type → GPU')
    print('   Training will be very slow on CPU.')

## 🔑 Step 2: Kaggle API Setup

To download datasets, you need a Kaggle account:
1. Go to https://www.kaggle.com/settings/account
2. Click **'Create New Token'** → downloads `kaggle.json`
3. Upload `kaggle.json` when prompted below

In [ ]:
import os
from google.colab import files

# Upload kaggle.json
print('📂 Upload your kaggle.json file:')
uploaded = files.upload()

# Set up Kaggle credentials
os.makedirs('/root/.config/kaggle', exist_ok=True)
!cp kaggle.json /root/.config/kaggle/
!chmod 600 /root/.config/kaggle/kaggle.json
print('✅ Kaggle credentials configured')

## 📁 Step 3: Download Training Datasets

We use **two freely available datasets**:
- **CIFAKE** (120k images: 60k real CIFAR-10 + 60k Stable Diffusion) — for AI image detection
- **140k Real and Fake Faces** — for deepfake/GAN face detection

In [ ]:
import zipfile
from pathlib import Path

DATA_DIR = Path('/content/data')
DATA_DIR.mkdir(exist_ok=True)

print('📥 Downloading CIFAKE dataset (Real vs AI-Generated)...')
!kaggle datasets download -d birdy654/cifake-real-and-ai-generated-synthetic-images -p {DATA_DIR}/cifake --unzip -q
print('✅ CIFAKE downloaded')

print('\n📥 Downloading 140k Real and Fake Faces dataset...')
!kaggle datasets download -d xhlulu/140k-real-and-fake-faces -p {DATA_DIR}/faces --unzip -q
print('✅ 140k Faces downloaded')

# Verify downloads
for split in ['train', 'test']:
    for label in ['REAL', 'FAKE']:
        path = DATA_DIR / 'cifake' / split / label
        if path.exists():
            count = len(list(path.glob('*.jpg')) + list(path.glob('*.png')))
            print(f'  CIFAKE {split}/{label}: {count:,} images')

print('\n✅ All datasets ready!')

## 🏗️ Step 4: Data Pipeline

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from PIL import Image
import numpy as np
import albumentations as A
from albumentations.pytorch import ToTensorV2
from pathlib import Path

# ─── Image Size & Normalization (ImageNet standard) ───────────────────────────
IMG_SIZE = 224
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

# ─── Augmentation Pipeline ────────────────────────────────────────────────────
train_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
    A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, p=0.3),
    A.GaussianBlur(blur_limit=(3, 5), p=0.2),
    A.GaussNoise(var_limit=(5, 30), p=0.2),
    A.JPEG(quality_lower=70, quality_upper=100, p=0.3),  # Simulate JPEG artifacts
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2(),
])

# ─── Dataset Class ────────────────────────────────────────────────────────────
class DeepfakeDataset(Dataset):
    """Binary image dataset: 0=REAL, 1=FAKE"""
    
    def __init__(self, real_dir: Path, fake_dir: Path, transform=None, max_per_class: int = None):
        self.transform = transform
        real_exts = {'.jpg', '.jpeg', '.png', '.webp'}
        
        real_imgs = [p for p in real_dir.rglob('*') if p.suffix.lower() in real_exts]
        fake_imgs = [p for p in fake_dir.rglob('*') if p.suffix.lower() in real_exts]
        
        if max_per_class:
            real_imgs = real_imgs[:max_per_class]
            fake_imgs = fake_imgs[:max_per_class]
        
        # Label: 0=real, 1=fake
        self.samples = [(p, 0) for p in real_imgs] + [(p, 1) for p in fake_imgs]
        np.random.shuffle(self.samples)
        
        print(f'  Dataset: {len(real_imgs):,} real + {len(fake_imgs):,} fake = {len(self.samples):,} total')
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = np.array(Image.open(path).convert('RGB'))
        if self.transform:
            img = self.transform(image=img)['image']
        return img, label

print('✅ Dataset class ready')

In [ ]:
# ─── Build CIFAKE Datasets ────────────────────────────────────────────────────
cifake_dir = DATA_DIR / 'cifake'

print('Building CIFAKE train dataset...')
train_dataset = DeepfakeDataset(
    real_dir=cifake_dir / 'train' / 'REAL',
    fake_dir=cifake_dir / 'train' / 'FAKE',
    transform=train_transform,
)

print('Building CIFAKE val dataset...')
val_dataset = DeepfakeDataset(
    real_dir=cifake_dir / 'test' / 'REAL',
    fake_dir=cifake_dir / 'test' / 'FAKE',
    transform=val_transform,
)

BATCH_SIZE = 64
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=4, pin_memory=True)

print(f'\n✅ Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')

## 🧠 Step 5: Model 1 — EfficientNet-B5 Deepfake Detector

In [ ]:
import timm

def build_efficientnet_b5(num_classes: int = 2, pretrained: bool = True) -> nn.Module:
    """
    EfficientNet-B5 with ImageNet pretrained weights + binary classification head.
    Matches the architecture in MediaTruth's deepfake_detector.py exactly.
    """
    model = timm.create_model(
        'efficientnet_b5',
        pretrained=pretrained,
        num_classes=num_classes,
    )
    
    # Freeze backbone for first few epochs (transfer learning)
    for name, param in model.named_parameters():
        if 'classifier' not in name:
            param.requires_grad = False
    
    return model

efficientnet = build_efficientnet_b5().to(device)

# Count parameters
total = sum(p.numel() for p in efficientnet.parameters())
trainable = sum(p.numel() for p in efficientnet.parameters() if p.requires_grad)
print(f'✅ EfficientNet-B5 built')
print(f'   Total parameters: {total/1e6:.1f}M')
print(f'   Trainable (head only): {trainable/1e3:.1f}K')

In [ ]:
import time
from sklearn.metrics import accuracy_score, roc_auc_score

def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, all_preds, all_labels = 0, [], []
    
    for batch_idx, (imgs, labels) in enumerate(loader):
        imgs, labels = imgs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        preds = outputs.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())
        
        if batch_idx % 100 == 0:
            print(f'  Batch {batch_idx}/{len(loader)}, Loss: {loss.item():.4f}', end='\r')
    
    acc = accuracy_score(all_labels, all_preds)
    return total_loss / len(loader), acc


@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    total_loss, all_probs, all_labels = 0, [], []
    
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        total_loss += loss.item()
        
        probs = torch.softmax(outputs, dim=1)[:, 1].cpu().numpy()
        all_probs.extend(probs)
        all_labels.extend(labels.cpu().numpy())
    
    preds = [1 if p > 0.5 else 0 for p in all_probs]
    acc = accuracy_score(all_labels, preds)
    auc = roc_auc_score(all_labels, all_probs)
    return total_loss / len(loader), acc, auc


print('✅ Training functions ready')

In [ ]:
import matplotlib.pyplot as plt

# ─── Training Config ──────────────────────────────────────────────────────────
EPOCHS_PHASE1 = 3   # Phase 1: Train head only (frozen backbone)
EPOCHS_PHASE2 = 7   # Phase 2: Fine-tune full model
LR_PHASE1 = 1e-3
LR_PHASE2 = 1e-4
WEIGHTS_DIR = Path('/content/weights')
WEIGHTS_DIR.mkdir(exist_ok=True)

criterion = nn.CrossEntropyLoss()
history = {'train_loss': [], 'val_loss': [], 'val_acc': [], 'val_auc': []}

# ═══════════════════════════════════════════════════════════
# PHASE 1: Train classification head only
# ═══════════════════════════════════════════════════════════
print('=' * 55)
print('PHASE 1: Training classification head (frozen backbone)')
print('=' * 55)

optimizer_p1 = optim.AdamW(
    filter(lambda p: p.requires_grad, efficientnet.parameters()),
    lr=LR_PHASE1, weight_decay=1e-4
)

best_auc = 0.0
for epoch in range(1, EPOCHS_PHASE1 + 1):
    t0 = time.time()
    train_loss, train_acc = train_epoch(efficientnet, train_loader, criterion, optimizer_p1, device)
    val_loss, val_acc, val_auc = validate(efficientnet, val_loader, criterion, device)
    
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['val_auc'].append(val_auc)
    
    print(f'Epoch {epoch:02d}/{EPOCHS_PHASE1} | '
          f'TrainLoss={train_loss:.4f} TrainAcc={train_acc:.4f} | '
          f'ValLoss={val_loss:.4f} ValAcc={val_acc:.4f} ValAUC={val_auc:.4f} | '
          f'{time.time()-t0:.0f}s')
    
    if val_auc > best_auc:
        best_auc = val_auc
        torch.save(efficientnet.state_dict(), WEIGHTS_DIR / 'efficientnet_b5_deepfake.pth')
        print(f'  💾 Saved best model (AUC={best_auc:.4f})')

# ═══════════════════════════════════════════════════════════
# PHASE 2: Fine-tune full model with lower LR
# ═══════════════════════════════════════════════════════════
print('\n' + '=' * 55)
print('PHASE 2: Fine-tuning full model (unfrozen backbone)')
print('=' * 55)

# Unfreeze all parameters
for param in efficientnet.parameters():
    param.requires_grad = True

optimizer_p2 = optim.AdamW(efficientnet.parameters(), lr=LR_PHASE2, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer_p2, T_max=EPOCHS_PHASE2)

for epoch in range(1, EPOCHS_PHASE2 + 1):
    t0 = time.time()
    train_loss, train_acc = train_epoch(efficientnet, train_loader, criterion, optimizer_p2, device)
    val_loss, val_acc, val_auc = validate(efficientnet, val_loader, criterion, device)
    scheduler.step()
    
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['val_auc'].append(val_auc)
    
    print(f'Epoch {epoch:02d}/{EPOCHS_PHASE2} | '
          f'TrainLoss={train_loss:.4f} TrainAcc={train_acc:.4f} | '
          f'ValLoss={val_loss:.4f} ValAcc={val_acc:.4f} ValAUC={val_auc:.4f} | '
          f'LR={scheduler.get_last_lr()[0]:.2e} | {time.time()-t0:.0f}s')
    
    if val_auc > best_auc:
        best_auc = val_auc
        torch.save(efficientnet.state_dict(), WEIGHTS_DIR / 'efficientnet_b5_deepfake.pth')
        print(f'  💾 Saved best model (AUC={best_auc:.4f})')

print(f'\n🎯 EfficientNet-B5 training complete! Best AUC: {best_auc:.4f}')

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

epochs = range(1, len(history['train_loss']) + 1)
axes[0].plot(epochs, history['train_loss'], 'b-', label='Train')
axes[0].plot(epochs, history['val_loss'], 'r-', label='Val')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(True)

axes[1].plot(epochs, history['val_acc'], 'g-')
axes[1].set_title('Val Accuracy'); axes[1].grid(True)
axes[1].axhline(y=0.9, color='r', linestyle='--', label='90% target')
axes[1].legend()

axes[2].plot(epochs, history['val_auc'], 'm-')
axes[2].set_title('Val AUC-ROC'); axes[2].grid(True)
axes[2].axhline(y=0.95, color='r', linestyle='--', label='0.95 target')
axes[2].legend()

plt.suptitle(f'EfficientNet-B5 Deepfake Detector — Best AUC: {best_auc:.4f}', fontsize=12)
plt.tight_layout()
plt.savefig(WEIGHTS_DIR / 'efficientnet_training_curves.png', dpi=120)
plt.show()
print('✅ Training curves saved')

## 🧠 Step 6: Model 2 — ResNet-50 GAN Detector (CNNDetect architecture)

In [ ]:
import torchvision.models as tv

def build_cnn_detect(pretrained: bool = True) -> nn.Module:
    """
    ResNet-50 with binary classification head.
    Matches MediaTruth's gan_detector.py architecture exactly.
    Uses sigmoid output (not softmax) — score = sigmoid(logit).
    """
    model = tv.resnet50(weights=tv.ResNet50_Weights.IMAGENET1K_V2 if pretrained else None)
    model.fc = nn.Linear(model.fc.in_features, 1)  # Binary: real vs GAN
    return model

cnn_detect = build_cnn_detect().to(device)

total = sum(p.numel() for p in cnn_detect.parameters())
print(f'✅ CNNDetect (ResNet-50) built')
print(f'   Total parameters: {total/1e6:.1f}M')

In [ ]:
def train_epoch_binary(model, loader, criterion, optimizer, device):
    """Training loop for binary (sigmoid) output models."""
    model.train()
    total_loss, all_preds, all_labels = 0, [], []
    
    for batch_idx, (imgs, labels) in enumerate(loader):
        imgs = imgs.to(device)
        labels_f = labels.float().unsqueeze(1).to(device)
        
        optimizer.zero_grad()
        logits = model(imgs)
        loss = criterion(logits, labels_f)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        total_loss += loss.item()
        probs = torch.sigmoid(logits).squeeze().cpu().detach().numpy()
        preds = (probs > 0.5).astype(int)
        all_preds.extend(preds.flatten())
        all_labels.extend(labels.numpy())
        
        if batch_idx % 100 == 0:
            print(f'  Batch {batch_idx}/{len(loader)}, Loss: {loss.item():.4f}', end='\r')
    
    acc = accuracy_score(all_labels, all_preds)
    return total_loss / len(loader), acc


@torch.no_grad()
def validate_binary(model, loader, criterion, device):
    model.eval()
    total_loss, all_probs, all_labels = 0, [], []
    
    for imgs, labels in loader:
        imgs = imgs.to(device)
        labels_f = labels.float().unsqueeze(1).to(device)
        logits = model(imgs)
        loss = criterion(logits, labels_f)
        total_loss += loss.item()
        
        probs = torch.sigmoid(logits).squeeze().cpu().numpy()
        all_probs.extend(probs.flatten())
        all_labels.extend(labels.numpy())
    
    preds = [1 if p > 0.5 else 0 for p in all_probs]
    acc = accuracy_score(all_labels, preds)
    auc = roc_auc_score(all_labels, all_probs)
    return total_loss / len(loader), acc, auc


print('✅ Binary training functions ready')

In [ ]:
# ─── Train CNNDetect GAN Detector ─────────────────────────────────────────────
EPOCHS_GAN = 10
LR_GAN = 1e-4

criterion_bce = nn.BCEWithLogitsLoss()
optimizer_gan = optim.AdamW(cnn_detect.parameters(), lr=LR_GAN, weight_decay=1e-4)
scheduler_gan = optim.lr_scheduler.CosineAnnealingLR(optimizer_gan, T_max=EPOCHS_GAN)

print('=' * 55)
print('Training CNNDetect GAN Detector (ResNet-50)')
print('=' * 55)

gan_history = {'train_loss': [], 'val_loss': [], 'val_acc': [], 'val_auc': []}
best_auc_gan = 0.0

for epoch in range(1, EPOCHS_GAN + 1):
    t0 = time.time()
    train_loss, train_acc = train_epoch_binary(cnn_detect, train_loader, criterion_bce, optimizer_gan, device)
    val_loss, val_acc, val_auc = validate_binary(cnn_detect, val_loader, criterion_bce, device)
    scheduler_gan.step()
    
    gan_history['train_loss'].append(train_loss)
    gan_history['val_loss'].append(val_loss)
    gan_history['val_acc'].append(val_acc)
    gan_history['val_auc'].append(val_auc)
    
    print(f'Epoch {epoch:02d}/{EPOCHS_GAN} | '
          f'TrainLoss={train_loss:.4f} TrainAcc={train_acc:.4f} | '
          f'ValLoss={val_loss:.4f} ValAcc={val_acc:.4f} ValAUC={val_auc:.4f} | '
          f'LR={scheduler_gan.get_last_lr()[0]:.2e} | {time.time()-t0:.0f}s')
    
    if val_auc > best_auc_gan:
        best_auc_gan = val_auc
        torch.save(cnn_detect.state_dict(), WEIGHTS_DIR / 'cnn_detect.pth')
        print(f'  💾 Saved best GAN detector (AUC={best_auc_gan:.4f})')

print(f'\n🎯 CNNDetect training complete! Best AUC: {best_auc_gan:.4f}')

## ✅ Step 7: Verify & Export Models

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# ─── Final Evaluation ─────────────────────────────────────────────────────────
print('=' * 55)
print('FINAL EVALUATION — MediaTruth Production Models')
print('=' * 55)

# Load best saved weights
efficientnet.load_state_dict(torch.load(WEIGHTS_DIR / 'efficientnet_b5_deepfake.pth', weights_only=True))
cnn_detect.load_state_dict(torch.load(WEIGHTS_DIR / 'cnn_detect.pth', weights_only=True))
efficientnet.eval(); cnn_detect.eval()

@torch.no_grad()
def get_predictions(model, loader, binary=False):
    all_probs, all_labels = [], []
    for imgs, labels in loader:
        imgs = imgs.to(device)
        out = model(imgs)
        if binary:
            probs = torch.sigmoid(out).squeeze().cpu().numpy()
        else:
            probs = torch.softmax(out, dim=1)[:, 1].cpu().numpy()
        all_probs.extend(probs.flatten())
        all_labels.extend(labels.numpy())
    return np.array(all_probs), np.array(all_labels)

# EfficientNet evaluation
print('\n1. EfficientNet-B5 Deepfake Detector:')
probs_eff, labels_val = get_predictions(efficientnet, val_loader, binary=False)
preds_eff = (probs_eff > 0.5).astype(int)
print(classification_report(labels_val, preds_eff, target_names=['REAL', 'FAKE']))
auc_eff = roc_auc_score(labels_val, probs_eff)
print(f'AUC-ROC: {auc_eff:.4f}')

# CNNDetect evaluation
print('\n2. CNNDetect GAN Detector:')
probs_cnn, _ = get_predictions(cnn_detect, val_loader, binary=True)
preds_cnn = (probs_cnn > 0.5).astype(int)
print(classification_report(labels_val, preds_cnn, target_names=['REAL', 'FAKE']))
auc_cnn = roc_auc_score(labels_val, probs_cnn)
print(f'AUC-ROC: {auc_cnn:.4f}')

# Summary
print('\n' + '=' * 55)
print('SUMMARY')
print(f'EfficientNet-B5 accuracy: {accuracy_score(labels_val, preds_eff)*100:.2f}%  AUC: {auc_eff:.4f}')
print(f'CNNDetect accuracy:       {accuracy_score(labels_val, preds_cnn)*100:.2f}%  AUC: {auc_cnn:.4f}')
print('=' * 55)

In [ ]:
# Verify .pth files are valid and list sizes
import os

print('📦 Model Weight Files:')
for fname in ['efficientnet_b5_deepfake.pth', 'cnn_detect.pth']:
    fpath = WEIGHTS_DIR / fname
    size_mb = fpath.stat().st_size / 1024 / 1024
    
    # Verify loadable
    state = torch.load(fpath, weights_only=True, map_location='cpu')
    print(f'  ✅ {fname}: {size_mb:.1f} MB, {len(state)} layers')

print('\n🎉 All models trained and verified!')
print('\n📋 Next steps:')
print('  1. Download the .pth files (next cell)')
print('  2. Copy them to MediaTruth/backend/models/weights/')
print('  3. Restart the backend — models auto-activate')

In [ ]:
# ─── Download Model Weights ────────────────────────────────────────────────────
from google.colab import files

print('📥 Downloading model weights...')
files.download(str(WEIGHTS_DIR / 'efficientnet_b5_deepfake.pth'))
files.download(str(WEIGHTS_DIR / 'cnn_detect.pth'))
files.download(str(WEIGHTS_DIR / 'efficientnet_training_curves.png'))
print('\n✅ Download started! Place .pth files in MediaTruth/backend/models/weights/')

## 🔧 Step 8: [OPTIONAL] Train on 140k Faces for Better Face Deepfake Detection

CIFAKE uses general images. If you want **better face-deepfake accuracy**, run this additional fine-tuning step on real/fake face images.

In [ ]:
# Build 140k Faces dataset
faces_dir = DATA_DIR / 'faces'

# Check the directory structure
for p in sorted(faces_dir.rglob('*'))[:20]:
    if p.is_dir():
        count = len(list(p.glob('*.jpg')) + list(p.glob('*.png')))
        print(f'{p.relative_to(faces_dir)}: {count:,} images')

In [ ]:
# Adapt paths based on actual directory structure found above
# Common structures for 140k Faces dataset:
real_faces_dir = faces_dir / 'real_vs_fake' / 'real-vs-fake' / 'train' / 'real'
fake_faces_dir = faces_dir / 'real_vs_fake' / 'real-vs-fake' / 'train' / 'fake'

if not real_faces_dir.exists():
    # Try alternative path
    real_faces_dir = faces_dir / 'train' / 'real'
    fake_faces_dir = faces_dir / 'train' / 'fake'

if real_faces_dir.exists() and fake_faces_dir.exists():
    print('Building 140k Faces fine-tuning dataset...')
    faces_train = DeepfakeDataset(
        real_dir=real_faces_dir,
        fake_dir=fake_faces_dir,
        transform=train_transform,
        max_per_class=20000,  # Use 20k per class for fine-tuning
    )
    faces_loader = DataLoader(faces_train, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=4, pin_memory=True)
    
    # Fine-tune EfficientNet on faces
    print('\nFine-tuning EfficientNet-B5 on face dataset...')
    optimizer_faces = optim.AdamW(efficientnet.parameters(), lr=5e-5, weight_decay=1e-4)
    
    for epoch in range(1, 4):  # 3 more epochs on faces
        t0 = time.time()
        train_loss, train_acc = train_epoch(efficientnet, faces_loader, criterion, optimizer_faces, device)
        print(f'Face fine-tune Epoch {epoch}/3 | Loss={train_loss:.4f} Acc={train_acc:.4f} | {time.time()-t0:.0f}s')
    
    # Re-validate and save if improved
    _, face_val_acc, face_val_auc = validate(efficientnet, val_loader, criterion, device)
    print(f'\nPost-face-tuning: ValAcc={face_val_acc:.4f} ValAUC={face_val_auc:.4f}')
    
    if face_val_auc >= best_auc:
        torch.save(efficientnet.state_dict(), WEIGHTS_DIR / 'efficientnet_b5_deepfake.pth')
        print(f'✅ Model improved! Saved updated weights.')
    else:
        print('ℹ️  Face fine-tuning did not improve AUC. Keeping CIFAKE-trained weights.')
else:
    print('⚠️  140k Faces directory structure not found. Check paths above. Skipping face fine-tuning.')

## 📊 Step 9: Final Model Card

This cell generates a model card documenting the trained models.

In [ ]:
from datetime import datetime

model_card = f"""
# MediaTruth Model Card
Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}

## Model 1: EfficientNet-B5 Deepfake Detector
- **File:** efficientnet_b5_deepfake.pth
- **Architecture:** EfficientNet-B5 (timm) + Linear(2048, 2) head
- **Training Dataset:** CIFAKE (60k real + 60k AI-generated)
- **Input:** 224x224 RGB, ImageNet normalization
- **Output:** softmax over [real, fake], score = prob[1]
- **Val Accuracy:** {accuracy_score(labels_val, preds_eff)*100:.2f}%
- **Val AUC-ROC:** {auc_eff:.4f}

## Model 2: CNNDetect GAN Detector
- **File:** cnn_detect.pth
- **Architecture:** ResNet-50 (torchvision) + Linear(2048, 1) head
- **Training Dataset:** CIFAKE (60k real + 60k AI-generated)
- **Input:** 224x224 RGB, ImageNet normalization
- **Output:** sigmoid scalar, score = sigmoid(logit)
- **Val Accuracy:** {accuracy_score(labels_val, preds_cnn)*100:.2f}%
- **Val AUC-ROC:** {auc_cnn:.4f}

## Integration
Copy both .pth files to: MediaTruth/backend/models/weights/
Restart backend — models auto-activate (no code changes needed).
"""

print(model_card)

with open(WEIGHTS_DIR / 'MODEL_CARD.txt', 'w') as f:
    f.write(model_card)

files.download(str(WEIGHTS_DIR / 'MODEL_CARD.txt'))
print('✅ Model card saved and downloaded')